### Attempt 2

In [ ]:
"""
Add four vertical (z-normal) lateral walls to an equilibrated gel slab.

This script is fully consistent with the following LAMMPS group definitions:

    group polymer type 1 2
    group solvent type 3
    group support type 4
    group piston  type 5
    group walls   type 6
    group mobile  type 1 2 3
    group gel_system type 1 2 4 5 6

What this script does:
- Rotates ONLY the mobile group (types 1,2,3) in the x–y plane so the gel slab
  edges are square with the piston/support (types 4,5 are not rotated)
- Determines gel lateral extent from polymer beads with a clearance margin
- Removes solvent outside the lateral walls (keeps solvent above/below the gel)
- Generates four vertical walls (type 6) normal to z
- Writes a new LAMMPS data file
"""

import numpy as np
from scipy.spatial import KDTree


# =========================
# Group-consistent settings
# =========================
POLYMER_TYPES = {1, 2}        # group polymer
SOLVENT_TYPES = {3}           # group solvent
SUPPORT_TYPES = {4}           # group support
PISTON_TYPES  = {5}           # group piston
WALL_TYPE     = 6             # group walls

ROTATE_TYPES  = POLYMER_TYPES | SOLVENT_TYPES   # group mobile
FROZEN_TYPES  = SUPPORT_TYPES | PISTON_TYPES | {WALL_TYPE}  # all rigid bodies
WALL_CLEARANCE = 0.2                            # gap between gel and walls


# =========================
# Geometry utilities
# =========================
from scipy.spatial import ConvexHull

def find_mabr_rotation_angle(atoms):
    """
    Find rotation angle via minimum-area bounding rectangle of the convex hull.
    Robust for nearly-square gels where PCA fails.
    """
    poly = [a for a in atoms if a['type'] in POLYMER_TYPES]
    if not poly:
        raise ValueError("No polymer atoms found")

    xy = np.array([[a['x'], a['y']] for a in poly])
    hull = ConvexHull(xy)
    hull_pts = xy[hull.vertices]

    best_angle = 0.0
    best_area = np.inf

    # Test rotation angles defined by each hull edge
    for i in range(len(hull_pts)):
        edge = hull_pts[(i + 1) % len(hull_pts)] - hull_pts[i]
        angle = np.arctan2(edge[1], edge[0])

        c, s = np.cos(-angle), np.sin(-angle)
        rotated = hull_pts @ np.array([[c, -s], [s, c]]).T

        width = rotated[:, 0].ptp()
        height = rotated[:, 1].ptp()
        area = width * height

        if area < best_area:
            best_area = area
            best_angle = angle

    # Normalize to [-45°, +45°] so we pick the smallest rotation
    while best_angle > np.pi / 4:
        best_angle -= np.pi / 2
    while best_angle < -np.pi / 4:
        best_angle += np.pi / 2

    aspect = max(1e-12, best_area)  # just for reporting
    print(f"  MABR rotation angle: {np.degrees(best_angle):.2f}°")
    return best_angle, 1.0


def rotate_polymer_and_solvent(atoms):
    """
    Rotate mobile group (types 1,2,3) to align gel principal axes with box axes.
    """
    poly = [a for a in atoms if a['type'] in POLYMER_TYPES]
    if not poly:
        raise ValueError("No polymer atoms found")

    xy = np.array([[a['x'], a['y']] for a in poly])
    com = xy.mean(axis=0)

    angle, aspect_ratio = find_mabr_rotation_angle(atoms)

    print(f"Slab rotation angle (PCA): {np.degrees(angle):.2f} degrees "
          f"(aspect ratio: {aspect_ratio:.3f})")

    # Skip rotation if angle is effectively zero
    if abs(angle) < 1e-10 or abs(abs(angle) - np.pi) < 1e-10:
        print("  No rotation needed (angle ≈ 0° or 180°)")
        return atoms

    c, s = np.cos(-angle), np.sin(-angle)

    for a in atoms:
        if a['type'] in ROTATE_TYPES:
            x = a['x'] - com[0]
            y = a['y'] - com[1]
            a['x'] = c*x - s*y + com[0]
            a['y'] = s*x + c*y + com[1]

    return atoms


def remove_rogue_polymer(atoms, percentile=0.5):
    """Remove polymer beads that are outliers in x-y plane."""
    poly = [a for a in atoms if a['type'] in POLYMER_TYPES]
    xs = np.array([a['x'] for a in poly])
    ys = np.array([a['y'] for a in poly])
    
    xmin, xmax = np.percentile(xs, percentile), np.percentile(xs, 100 - percentile)
    ymin, ymax = np.percentile(ys, percentile), np.percentile(ys, 100 - percentile)
    
    kept = []
    removed = 0
    for a in atoms:
        if a['type'] in POLYMER_TYPES:
            if a['x'] < xmin or a['x'] > xmax or a['y'] < ymin or a['y'] > ymax:
                removed += 1
                continue
        kept.append(a)
    
    print(f"Removed {removed} rogue polymer atoms")
    return kept

def find_gel_extent(atoms, clearance, percentile=0.05):
    """Gel extent from polymer beads, using percentiles to exclude rogues."""
    poly = [a for a in atoms if a['type'] in POLYMER_TYPES]
    
    xs = np.array([a['x'] for a in poly])
    ys = np.array([a['y'] for a in poly])
    zs = np.array([a['z'] for a in poly])

    # Use percentiles to exclude outliers
    xmin = np.percentile(xs, percentile)
    xmax = np.percentile(xs, 100 - percentile)
    ymin = np.percentile(ys, percentile)
    ymax = np.percentile(ys, 100 - percentile)

    print(f"Polymer x range: {xmin:.2f} to {xmax:.2f} (width: {xmax-xmin:.2f})")
    print(f"Polymer y range: {ymin:.2f} to {ymax:.2f} (width: {ymax-ymin:.2f})")

    return {
        'xmin': xmin - clearance,
        'xmax': xmax + clearance,
        'ymin': ymin - clearance,
        'ymax': ymax + clearance,
        'zmin': zs.min(),
        'zmax': zs.max()
    }


def generate_wall_atoms(ext, box, spacing=0.2):
    """Generate four vertical walls (type 6) as single-layer triangular lattices."""
    walls = []
    
    zlo, zhi = box['zlo'], box['zhi']
    
    # For triangular lattice: rows are offset by spacing/2, row spacing is spacing*sqrt(3)/2
    row_spacing = spacing * np.sqrt(3) / 2
    
    # ±x walls (constant x, triangular lattice in y-z plane)
    for x in [ext['xmin'], ext['xmax']]:
        ny = int(np.ceil((ext['ymax'] - ext['ymin']) / spacing)) + 1
        nz = int(np.ceil((zhi - zlo) / row_spacing)) + 1
        
        for iz in range(nz):
            z = zlo + iz * row_spacing
            if z > zhi:
                continue
                
            # Alternate rows offset by spacing/2
            y_offset = (spacing / 2) if (iz % 2) else 0
            y_start = ext['ymin'] + y_offset
            
            y = y_start
            while y <= ext['ymax']:
                walls.append({'type': WALL_TYPE, 'x': x, 'y': y, 'z': z})
                y += spacing
    
    # ±y walls (constant y, triangular lattice in x-z plane)
    for y in [ext['ymin'], ext['ymax']]:
        nx = int(np.ceil((ext['xmax'] - ext['xmin']) / spacing)) + 1
        nz = int(np.ceil((zhi - zlo) / row_spacing)) + 1
        
        for iz in range(nz):
            z = zlo + iz * row_spacing
            if z > zhi:
                continue
                
            # Alternate rows offset by spacing/2
            x_offset = (spacing / 2) if (iz % 2) else 0
            x_start = ext['xmin'] + x_offset
            
            x = x_start
            while x <= ext['xmax']:
                walls.append({'type': WALL_TYPE, 'x': x, 'y': y, 'z': z})
                x += spacing
    
    print(f"Generated {len(walls)} wall atoms with triangular packing (spacing={spacing})")
    return walls



def remove_external_solvent(atoms, ext):
    """Remove solvent outside lateral wall bounds."""
    kept = []
    removed = 0

    for a in atoms:
        if a['type'] in SOLVENT_TYPES:
            if (a['x'] < ext['xmin'] or a['x'] > ext['xmax'] or
                a['y'] < ext['ymin'] or a['y'] > ext['ymax']):
                removed += 1
                continue
        kept.append(a)

    print(f"Removed {removed} solvent atoms")
    return kept

def remove_external_polymer(atoms, ext):
    """Remove polymer atoms outside wall boundaries."""
    kept = []
    removed = 0
    
    for a in atoms:
        if a['type'] in POLYMER_TYPES:
            if (a['x'] < ext['xmin'] or a['x'] > ext['xmax'] or
                a['y'] < ext['ymin'] or a['y'] > ext['ymax']):
                removed += 1
                continue
        kept.append(a)
    
    print(f"Removed {removed} polymer atoms outside walls")
    return kept


def remove_overlapping_solvent(atoms, wall_atoms, cutoff=1.5):
    """
    Remove solvent atoms that overlap with wall/support/piston atoms.
    Uses a KDTree for efficient spatial queries over large atom counts.
    
    Parameters:
    - atoms: existing atom list (may contain support/piston already)
    - wall_atoms: list of new wall atom dicts (not yet appended to atoms)
    - cutoff: minimum allowed distance between solvent and any rigid atom (σ)
    """
    # Collect ALL rigid-body positions: existing support + piston + new walls
    rigid_positions = []
    for a in atoms:
        if a['type'] in FROZEN_TYPES:
            rigid_positions.append([a['x'], a['y'], a['z']])
    for w in wall_atoms:
        rigid_positions.append([w['x'], w['y'], w['z']])
    
    if not rigid_positions:
        print("No rigid atoms found — skipping overlap removal")
        return atoms
    
    rigid_positions = np.array(rigid_positions)
    tree = KDTree(rigid_positions)
    print(f"Built KDTree with {len(rigid_positions)} rigid-body positions "
          f"(support + piston + walls)")
    
    # Query each solvent atom against the tree
    kept = []
    removed = 0
    for a in atoms:
        if a['type'] in SOLVENT_TYPES:
            pos = np.array([a['x'], a['y'], a['z']])
            dist, _ = tree.query(pos)
            if dist < cutoff:
                removed += 1
                continue
        kept.append(a)
    
    print(f"Removed {removed} solvent atoms within {cutoff}σ of rigid atoms")
    return kept

# =========================
# LAMMPS I/O
# =========================
def parse_lammps_data(filename):
    """Parse LAMMPS data file and extract all information."""
    
    atoms = []
    bonds = []
    box_bounds = {}
    masses = {}
    header_info = {}
    
    with open(filename, 'r') as f:
        lines = f.readlines()
    
    i = 0
    while i < len(lines):
        line = lines[i].strip()
        
        # Parse header info
        if 'atoms' in line and 'atom' not in line.lower().split()[0]:
            header_info['natoms'] = int(line.split()[0])
        elif 'bonds' in line and 'bond' not in line.lower().split()[0]:
            header_info['nbonds'] = int(line.split()[0])
        elif 'atom types' in line:
            header_info['atom_types'] = int(line.split()[0])
        elif 'bond types' in line:
            header_info['bond_types'] = int(line.split()[0])
        elif 'xlo xhi' in line:
            parts = line.split()
            box_bounds['xlo'] = float(parts[0])
            box_bounds['xhi'] = float(parts[1])
        elif 'ylo yhi' in line:
            parts = line.split()
            box_bounds['ylo'] = float(parts[0])
            box_bounds['yhi'] = float(parts[1])
        elif 'zlo zhi' in line:
            parts = line.split()
            box_bounds['zlo'] = float(parts[0])
            box_bounds['zhi'] = float(parts[1])
        
        # Parse Masses section
        elif line == 'Masses':
            i += 2  # Skip blank line
            while i < len(lines) and lines[i].strip() and not lines[i].strip().startswith(('Atoms', 'Bonds', 'Pair')):
                parts = lines[i].split()
                if len(parts) >= 2:
                    try:
                        masses[int(parts[0])] = float(parts[1])
                    except ValueError:
                        pass
                i += 1
            continue
        
        # Parse Atoms section
        elif line == 'Atoms' or line.startswith('Atoms'):
            i += 2  # Skip blank line
            while i < len(lines) and lines[i].strip() and not lines[i].strip().startswith(('Bonds', 'Velocities', 'Pair')):
                parts = lines[i].split()
                if len(parts) >= 6:
                    try:
                        atoms.append({
                            'id': int(parts[0]),
                            'mol': int(parts[1]),
                            'type': int(parts[2]),
                            'x': float(parts[3]),
                            'y': float(parts[4]),
                            'z': float(parts[5])
                        })
                    except ValueError:
                        pass
                i += 1
            continue
        
        # Parse Bonds section
        elif line == 'Bonds' or line.startswith('Bonds'):
            i += 2  # Skip blank line
            while i < len(lines) and lines[i].strip():
                parts = lines[i].split()
                if len(parts) >= 4:
                    try:
                        bonds.append({
                            'id': int(parts[0]),
                            'type': int(parts[1]),
                            'atom1': int(parts[2]),
                            'atom2': int(parts[3])
                        })
                    except ValueError:
                        pass
                i += 1
            continue
        
        i += 1
    
    return atoms, bonds, box_bounds, masses


def write_lammps_data(filename, atoms, bonds, box, masses):
    old2new = {a['id']: i+1 for i,a in enumerate(atoms)}

    valid_bonds = [
        {'id': i+1, 'type': b['type'],
         'atom1': old2new[b['atom1']], 'atom2': old2new[b['atom2']]}
        for i,b in enumerate(bonds)
        if b['atom1'] in old2new and b['atom2'] in old2new
    ]

    with open(filename, 'w') as f:
        f.write("LAMMPS data file with lateral walls\n\n")
        f.write(f"{len(atoms)} atoms\n{len(valid_bonds)} bonds\n\n")
        f.write("6 atom types\n1 bond types\n\n")
        f.write(f"{box['xlo']} {box['xhi']} xlo xhi\n")
        f.write(f"{box['ylo']} {box['yhi']} ylo yhi\n")
        f.write(f"{box['zlo']} {box['zhi']} zlo zhi\n\n")

        f.write("Masses\n\n")
        for i in range(1,7):
            f.write(f"{i} {masses.get(i,1.0)}\n")

        f.write("\nAtoms\n\n")
        for i,a in enumerate(atoms,1):
            f.write(f"{i} {a['mol']} {a['type']} "
                    f"{a['x']:.6f} {a['y']:.6f} {a['z']:.6f}\n")

        if valid_bonds:
            f.write("\nBonds\n\n")
            for b in valid_bonds:
                f.write(f"{b['id']} {b['type']} {b['atom1']} {b['atom2']}\n")


# =========================
# Main driver
# =========================
def add_walls_to_slab(input_file, output_file, wall_spacing, cutoff):
    atoms, bonds, box, masses = parse_lammps_data(input_file)

    atoms = rotate_polymer_and_solvent(atoms)
    ext = find_gel_extent(atoms, WALL_CLEARANCE, percentile=0.1)
    
    # Remove atoms outside walls
    atoms = remove_external_polymer(atoms, ext)
    atoms = remove_external_solvent(atoms, ext)

    walls = generate_wall_atoms(ext, box, wall_spacing)

    # Remove solvent overlapping with walls/support/piston before appending walls
    atoms = remove_overlapping_solvent(atoms, walls, cutoff)

    max_id = max(a['id'] for a in atoms)
    max_mol = max(a['mol'] for a in atoms)

    for i, w in enumerate(walls):
        atoms.append({
            'id': max_id + i + 1,
            'mol': max_mol + i + 1,
            'type': WALL_TYPE,
            'x': w['x'], 'y': w['y'], 'z': w['z']
        })

    write_lammps_data(output_file, atoms, bonds, box, masses)
    print(f"Wrote {output_file}")



In [ ]:
# Inputs

input_file = "../../lammps_data_files_local/final_config_slab_support_5beads_tall_rho04_p1.5_1.0_1.0_600000.data"
output_file = "../../lammps_data_files_local/walled_slab_support_5beads_tall_rho04_p1.51_1.0_1.0_600000.data"
wall_spacing = 0.2
cutoff=0.8 # deleting solvent with 0.6σ from walls


add_walls_to_slab(input_file, output_file, wall_spacing, cutoff)

